In [1]:
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\SAGAR\.pyenv\pyenv-win\versions\3.8.10\python.exe -m pip install --upgrade pip' command.


In [2]:
import os
import json
import re
from typing import List, Dict, Any, Tuple
from pathlib import Path
from datetime import datetime
from fastapi.middleware.cors import CORSMiddleware
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import httpx
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

app = FastAPI(title="Indy ADHD Copilot API")

# Add CORS middleware to allow frontend to communicate
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "http://localhost:3001"],  # Next.js dev server
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

COLAB_LLM_URL = os.environ.get("COLAB_LLM_URL", "https://REPLACE_ME.trycloudflare.com")
print(f"⚠️ MAKE SURE TO UPDATE THIS URL FROM COLAB: {COLAB_LLM_URL}")

# Fine-tuned model server endpoint
NLP_SERVER_URL = "http://localhost:8001"
print(f"NLP Server: {NLP_SERVER_URL}")

SESSIONS_DIR = Path("sessions")
SESSIONS_DIR.mkdir(exist_ok=True)

LABEL_DESCRIPTIONS = {
    "affective": "expressing emotions, feelings, distress, sadness, fear, love, anger",
    "cognitive": "thinking, reasoning, reflecting, understanding, analyzing, wondering",
    "agency":    "taking action, making decisions, expressing control, planning, choosing"
}

# ─────────────────────────────────────────────
# SCHEMAS
# ─────────────────────────────────────────────

class ChatRequest(BaseModel):
    session_id: int
    message: str
    history: List[Dict[str, str]]

# ─────────────────────────────────────────────
# LOGIC HELPERS & RAG
# ─────────────────────────────────────────────

async def score_text(text: str):
    """
    Call the fine-tuned model server to score text.
    Returns dict with scores (0-1) for each category and dominant category.
    """
    try:
        async with httpx.AsyncClient() as client_http:
            response = await client_http.post(
                f"{NLP_SERVER_URL}/score",
                json={"text": text},
                timeout=10.0
            )
            response.raise_for_status()
            return response.json()
    except Exception as e:
        print(f"Error calling NLP server: {e}")
        # Graceful fallback
        return {"scores": {"affective": 0, "cognitive": 0, "agency": 0}, "dominant": "unknown"}

def build_rag_corpus():
    documents = []
    for path in sorted(SESSIONS_DIR.glob("session_*.json")):
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
                session_id = data.get("session_id", "Unknown")
                for msg in data.get("conversation", []):
                    content = msg.get("content", "")
                    notes = msg.get("notes", {})
                    if isinstance(notes, dict) and "notes" in notes:
                        for note in notes["notes"]:
                            content += f" | NOTE ({note.get('category')}): {note.get('title')} - {note.get('content')}"
                    if content.strip():
                        documents.append({
                            "session": session_id,
                            "role": msg.get("role", "user").capitalize(),
                            "text": content,
                            "timestamp": msg.get("timestamp", "")[:10]
                        })
        except Exception as e:
            print(f"Warning: could not read {path} for RAG: {e}")
    return documents

def get_relevant_context(current_message: str, top_k: int = 3):
    docs = build_rag_corpus()
    if not docs:
        return "No past session history available."
    
    texts = [d["text"] for d in docs]
    vectorizer = TfidfVectorizer(stop_words='english')
    try:
        tfidf_matrix = vectorizer.fit_transform(texts)
        query_vec = vectorizer.transform([current_message])
        similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
        
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        relevant_blocks = []
        for idx in top_indices:
            # threshold score so we don't pull completely unrelated things
            if similarities[idx] > 0.05:
                doc = docs[idx]
                relevant_blocks.append(f"[Session {doc['session']} - {doc['timestamp']}] {doc['role']}: {doc['text']}")
                
        if not relevant_blocks:
             return "No highly relevant past context found for this specific query."
        
        return "RELEVANT PAST CONTEXT:\n" + "\n".join(relevant_blocks)
    except Exception as e:
        print(f"RAG Error: {e}")
        return "Error retrieving history."

def load_or_create_session(session_id: int) -> dict:
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    else:
        return {
            "session_id": session_id,
            "created": datetime.now().isoformat(),
            "conversation": [],
            "cognitive_shifts": []
        }

def save_session(session_id: int, session_data: dict) -> None:
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    with open(path, "w") as f:
        json.dump(session_data, f, indent=2)

def extract_notes_from_response(raw_reply: str) -> Tuple[str, Dict[str, Any]]:
    notes: Dict[str, Any] = {}
    clean_text = raw_reply
    if "### UPDATED_NOTES" in raw_reply:
        clean_text = raw_reply.split("### UPDATED_NOTES")[0].strip()
        json_part = raw_reply.split("### UPDATED_NOTES")[-1].strip()
        try:
            json_part = re.sub(r"^```(?:json)?", "", json_part).strip()
            json_part = re.sub(r"```$", "", json_part).strip()
            json_match = re.search(r'\{.*\}', json_part, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                notes = json.loads(json_str)
            else:
                notes = json.loads(json_part)
        except json.JSONDecodeError as e:
            print(f"Warning: Failed to parse JSON notes: {e}")
            notes = {"error": "Failed to parse notes", "raw": json_part[:200]}
    else:
        try:
            json_match = re.search(r'\{[^{}]*"[^"]*"[^{}]*\}', raw_reply, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                parsed = json.loads(json_str)
                clean_text = raw_reply[:json_match.start()].strip()
                notes = parsed
        except:
            pass
    return clean_text, notes

# ─────────────────────────────────────────────
# ENDPOINTS
# ─────────────────────────────────────────────

@app.post("/chat")
async def chat_endpoint(req: ChatRequest):
    try:
        # 1. Cognitive Shift Analysis (call NLP server)
        analysis = await score_text(req.message)
        
        # 2. Get RAG Context (Semantic Search using TF-IDF)
        past_context = get_relevant_context(req.message)
        
        # 3. Construct System Prompt
        system_prompt = f"""You are Empathia, an empathetic, multilingual therapeutic assistant. You support code-switching (e.g., Hindi-English, Spanish-English) and always provide non-judgmental, validating responses.\n\nHere is some semantically relevant context pulled from past sessions for this user. You may use this context to inform your responses, but do not explicitly mention that you are reading from notes.\n\n{past_context}\n"""

        # 4. Call fine-tuned LLM on Google Colab
        messages = [{"role": "system", "content": system_prompt}] + req.history + [{"role": "user", "content": req.message}]
        
        try:
            async with httpx.AsyncClient(timeout=120.0) as client_http:
                response = await client_http.post(
                    f"{COLAB_LLM_URL}/generate",
                    json={"messages": messages, "max_new_tokens": 1024, "temperature": 0.7}
                )
                response.raise_for_status()
                raw_reply = response.json()["response"]
        except Exception as llm_error:
            print(f"Colab API Error: {llm_error}")
            raise HTTPException(status_code=502, detail=f"Colab API error. Check if the Colab notebook is running and URL is correct: {str(llm_error)}")
        
        # 5. Parsing & Cleaning
        clean_text, notes = extract_notes_from_response(raw_reply)
        
        if not notes:
            notes = {
                "session_id": req.session_id,
                "date": datetime.now().strftime("%Y-%m-%d"),
                "session_type": "brain_dump",
                "priorities": [],
                "blockers": [],
                "small_wins": [],
                "insights": []
            }

        # 6. Save session with conversation history
        session_data = load_or_create_session(req.session_id)
        session_data["conversation"].append({
            "role": "user",
            "content": req.message,
            "timestamp": datetime.now().isoformat()
        })
        session_data["conversation"].append({
            "role": "assistant",
            "content": clean_text,
            "timestamp": datetime.now().isoformat(),
            "cognitive_shift": analysis,
            "notes": notes
        })
        session_data["cognitive_shifts"].append({
            "message": req.message,
            "analysis": analysis,
            "timestamp": datetime.now().isoformat()
        })
        
        save_session(req.session_id, session_data)
        
        return {
            "reply": clean_text,
            "cognitive_shift": analysis,
            "extracted_notes": notes
        }

    except HTTPException:
        raise
    except Exception as e:
        print(f"Error in chat endpoint: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


Initializing LLM client...
NLP Server: http://localhost:8001


In [3]:

@app.get("/sessions/{session_id}/cognitive-shifts")
async def get_cognitive_shifts(session_id: int):
    """
    Retrieve cognitive shift data for all user messages in a session.
    Returns an array of shifts with message number, scores, and dominant category.
    """
    try:
        session_data = load_or_create_session(session_id)
        
        # Extract cognitive shifts from conversation history
        # Only include shifts for user messages
        shifts = []
        message_num = 0
        
        for i, msg in enumerate(session_data["conversation"]):
            if msg["role"] == "user":
                message_num += 1
                # Get the corresponding assistant response (should be next message)
                if i + 1 < len(session_data["conversation"]):
                    assistant_msg = session_data["conversation"][i + 1]
                    if "cognitive_shift" in assistant_msg:
                        shift_data = assistant_msg["cognitive_shift"]
                        shifts.append({
                            "message_num": message_num,
                            "affective": shift_data["scores"].get("affective", 0),
                            "cognitive": shift_data["scores"].get("cognitive", 0),
                            "agency": shift_data["scores"].get("agency", 0),
                            "dominant": shift_data.get("dominant", ""),
                            "timestamp": msg.get("timestamp", ""),
                            "content": msg.get("content", "")[:100]  # First 100 chars
                        })
        
        return {
            "session_id": session_id,
            "total_messages": message_num,
            "shifts": shifts
        }
    
    except Exception as e:
        print(f"Error retrieving cognitive shifts: {e}")
        raise HTTPException(status_code=500, detail=str(e))


In [4]:
import json

def merge_notes(existing_notes: dict, llm_response: str) -> dict:
    """
    Parses the UPDATED_NOTES JSON from the LLM response and merges it
    with the existing notes. The LLM is expected to return the full
    merged object, so this function simply validates and returns it.
    Falls back to existing notes if parsing fails.
    """
    marker = "### UPDATED_NOTES"
    if marker not in llm_response:
        return existing_notes

    raw_json = llm_response.split(marker, 1)[1].strip()

    try:
        updated = json.loads(raw_json)
        # Ensure the new notes dict is merged on top of existing notes
        # so no old keys are silently dropped if the LLM forgets them
        if "notes" in existing_notes and "notes" in updated:
            merged_notes = {**existing_notes.get("notes", {}), **updated["notes"]}
            updated["notes"] = merged_notes
        return updated
    except json.JSONDecodeError:
        # If the LLM returned malformed JSON, preserve what we had
        return existing_notes, 

In [5]:
import os
import json
import re
from typing import List, Dict, Any, Tuple
from pathlib import Path
from datetime import datetime
from fastapi.middleware.cors import CORSMiddleware
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import httpx
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

app = FastAPI(title="Indy ADHD Copilot API")

# Add CORS middleware to allow frontend to communicate
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "http://localhost:3001"],  # Next.js dev server
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

COLAB_LLM_URL = os.environ.get("COLAB_LLM_URL", "https://REPLACE_ME.trycloudflare.com")
print(f"⚠️ MAKE SURE TO UPDATE THIS URL FROM COLAB: {COLAB_LLM_URL}")

# Fine-tuned model server endpoint
NLP_SERVER_URL = "http://localhost:8001"
print(f"NLP Server: {NLP_SERVER_URL}")

SESSIONS_DIR = Path("sessions")
SESSIONS_DIR.mkdir(exist_ok=True)

LABEL_DESCRIPTIONS = {
    "affective": "expressing emotions, feelings, distress, sadness, fear, love, anger",
    "cognitive": "thinking, reasoning, reflecting, understanding, analyzing, wondering",
    "agency":    "taking action, making decisions, expressing control, planning, choosing"
}

# ─────────────────────────────────────────────
# SCHEMAS
# ─────────────────────────────────────────────

class ChatRequest(BaseModel):
    session_id: int
    message: str
    history: List[Dict[str, str]]

# ─────────────────────────────────────────────
# LOGIC HELPERS & RAG
# ─────────────────────────────────────────────

async def score_text(text: str):
    """
    Call the fine-tuned model server to score text.
    Returns dict with scores (0-1) for each category and dominant category.
    """
    try:
        async with httpx.AsyncClient() as client_http:
            response = await client_http.post(
                f"{NLP_SERVER_URL}/score",
                json={"text": text},
                timeout=10.0
            )
            response.raise_for_status()
            return response.json()
    except Exception as e:
        print(f"Error calling NLP server: {e}")
        # Graceful fallback
        return {"scores": {"affective": 0, "cognitive": 0, "agency": 0}, "dominant": "unknown"}

def build_rag_corpus():
    documents = []
    for path in sorted(SESSIONS_DIR.glob("session_*.json")):
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
                session_id = data.get("session_id", "Unknown")
                for msg in data.get("conversation", []):
                    content = msg.get("content", "")
                    notes = msg.get("notes", {})
                    if isinstance(notes, dict) and "notes" in notes:
                        for note in notes["notes"]:
                            content += f" | NOTE ({note.get('category')}): {note.get('title')} - {note.get('content')}"
                    if content.strip():
                        documents.append({
                            "session": session_id,
                            "role": msg.get("role", "user").capitalize(),
                            "text": content,
                            "timestamp": msg.get("timestamp", "")[:10]
                        })
        except Exception as e:
            print(f"Warning: could not read {path} for RAG: {e}")
    return documents

def get_relevant_context(current_message: str, top_k: int = 3):
    docs = build_rag_corpus()
    if not docs:
        return "No past session history available."
    
    texts = [d["text"] for d in docs]
    vectorizer = TfidfVectorizer(stop_words='english')
    try:
        tfidf_matrix = vectorizer.fit_transform(texts)
        query_vec = vectorizer.transform([current_message])
        similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
        
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        relevant_blocks = []
        for idx in top_indices:
            # threshold score so we don't pull completely unrelated things
            if similarities[idx] > 0.05:
                doc = docs[idx]
                relevant_blocks.append(f"[Session {doc['session']} - {doc['timestamp']}] {doc['role']}: {doc['text']}")
                
        if not relevant_blocks:
             return "No highly relevant past context found for this specific query."
        
        return "RELEVANT PAST CONTEXT:\n" + "\n".join(relevant_blocks)
    except Exception as e:
        print(f"RAG Error: {e}")
        return "Error retrieving history."

def load_or_create_session(session_id: int) -> dict:
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    else:
        return {
            "session_id": session_id,
            "created": datetime.now().isoformat(),
            "conversation": [],
            "cognitive_shifts": []
        }

def save_session(session_id: int, session_data: dict) -> None:
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    with open(path, "w") as f:
        json.dump(session_data, f, indent=2)

def extract_notes_from_response(raw_reply: str) -> Tuple[str, Dict[str, Any]]:
    notes: Dict[str, Any] = {}
    clean_text = raw_reply
    if "### UPDATED_NOTES" in raw_reply:
        clean_text = raw_reply.split("### UPDATED_NOTES")[0].strip()
        json_part = raw_reply.split("### UPDATED_NOTES")[-1].strip()
        try:
            json_part = re.sub(r"^```(?:json)?", "", json_part).strip()
            json_part = re.sub(r"```$", "", json_part).strip()
            json_match = re.search(r'\{.*\}', json_part, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                notes = json.loads(json_str)
            else:
                notes = json.loads(json_part)
        except json.JSONDecodeError as e:
            print(f"Warning: Failed to parse JSON notes: {e}")
            notes = {"error": "Failed to parse notes", "raw": json_part[:200]}
    else:
        try:
            json_match = re.search(r'\{[^{}]*"[^"]*"[^{}]*\}', raw_reply, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                parsed = json.loads(json_str)
                clean_text = raw_reply[:json_match.start()].strip()
                notes = parsed
        except:
            pass
    return clean_text, notes

# ─────────────────────────────────────────────
# ENDPOINTS
# ─────────────────────────────────────────────

@app.post("/chat")
async def chat_endpoint(req: ChatRequest):
    try:
        # 1. Cognitive Shift Analysis (call NLP server)
        analysis = await score_text(req.message)
        
        # 2. Get RAG Context (Semantic Search using TF-IDF)
        past_context = get_relevant_context(req.message)
        
        # 3. Construct System Prompt
        system_prompt = f"""You are Empathia, an empathetic, multilingual therapeutic assistant. You support code-switching (e.g., Hindi-English, Spanish-English) and always provide non-judgmental, validating responses.\n\nHere is some semantically relevant context pulled from past sessions for this user. You may use this context to inform your responses, but do not explicitly mention that you are reading from notes.\n\n{past_context}\n"""

        # 4. Call fine-tuned LLM on Google Colab
        messages = [{"role": "system", "content": system_prompt}] + req.history + [{"role": "user", "content": req.message}]
        
        try:
            async with httpx.AsyncClient(timeout=120.0) as client_http:
                response = await client_http.post(
                    f"{COLAB_LLM_URL}/generate",
                    json={"messages": messages, "max_new_tokens": 1024, "temperature": 0.7}
                )
                response.raise_for_status()
                raw_reply = response.json()["response"]
        except Exception as llm_error:
            print(f"Colab API Error: {llm_error}")
            raise HTTPException(status_code=502, detail=f"Colab API error. Check if the Colab notebook is running and URL is correct: {str(llm_error)}")
        
        # 5. Parsing & Cleaning
        clean_text, notes = extract_notes_from_response(raw_reply)
        
        if not notes:
            notes = {
                "session_id": req.session_id,
                "date": datetime.now().strftime("%Y-%m-%d"),
                "session_type": "brain_dump",
                "priorities": [],
                "blockers": [],
                "small_wins": [],
                "insights": []
            }

        # 6. Save session with conversation history
        session_data = load_or_create_session(req.session_id)
        session_data["conversation"].append({
            "role": "user",
            "content": req.message,
            "timestamp": datetime.now().isoformat()
        })
        session_data["conversation"].append({
            "role": "assistant",
            "content": clean_text,
            "timestamp": datetime.now().isoformat(),
            "cognitive_shift": analysis,
            "notes": notes
        })
        session_data["cognitive_shifts"].append({
            "message": req.message,
            "analysis": analysis,
            "timestamp": datetime.now().isoformat()
        })
        
        save_session(req.session_id, session_data)
        
        return {
            "reply": clean_text,
            "cognitive_shift": analysis,
            "extracted_notes": notes
        }

    except HTTPException:
        raise
    except Exception as e:
        print(f"Error in chat endpoint: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


You should consider upgrading via the 'c:\Users\SAGAR\.pyenv\pyenv-win\versions\3.8.10\python.exe -m pip install --upgrade pip' command.
INFO:     Started server process [30316]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Note: you may need to restart the kernel to use updated packages.
INFO:     127.0.0.1:62934 - "OPTIONS /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62934 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:61874 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:55925 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62950 - "GET /sessions/1/cognitive-shifts HTTP/1.1" 200 OK
INFO:     127.0.0.1:60696 - "POST /chat HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [30316]
